# Synthetic QA Generation from a Single PDF

**Input:** `Visa_Credit_Card_Agreement.pdf` (6 pages)  
**Output:** grounded synthetic question–answer pairs in JSONL and CSV, plus validation and summary files.

This notebook extracts page-aware text, chunks it, uses an OpenAI model to generate diverse QA pairs with **verbatim evidence quotes**, validates the evidence against the extracted text, deduplicates, and exports results. It also generates explicitly **unanswerable** questions (with null answers) to evaluate hallucination resistance. Model-generated questions and answers should be reviewed before use as a benchmark.

**Important:** The agreement refers to a separate *Account Opening Disclosure* for actual APRs and many fee amounts. The notebook instructs the model not to invent those missing values.


In [1]:
%pip install -q pymupdf openai pandas tqdm

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration and API key
Set `OPENAI_API_KEY` in your environment or enter it securely when prompted. API calls incur charges.

In [5]:
import os, json, re, time, hashlib, getpass
from pathlib import Path
from collections import Counter
import fitz
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

PDF_PATH = Path("data/documents/Visa_Credit_Card_Agreement.pdf")
if not PDF_PATH.exists():
    PDF_PATH = Path("Visa_Credit_Card_Agreement.pdf")
assert PDF_PATH.exists(), f"Missing PDF: {PDF_PATH}"
OUTPUT_DIR = Path("synthetic_qa_output")
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL = "gpt-4o"  # Change to gpt-4.1 for higher quality at higher cost
QUESTIONS_PER_CHUNK = 6
CHUNK_CHARS = 3200
OVERLAP_CHARS = 350
MAX_RETRIES = 3
GENERATE_UNANSWERABLE = True

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("Input:", PDF_PATH, "| Model:", MODEL)

Input: data/documents/Visa_Credit_Card_Agreement.pdf | Model: gpt-4o


## 2. Extract PDF text with page references

In [6]:
def extract_pages(path):
    doc = fitz.open(path)
    pages = []
    for index, page in enumerate(doc):
        text = page.get_text("text", sort=True)
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        pages.append({"page": index + 1, "text": text})
    doc.close()
    return pages

pages = extract_pages(PDF_PATH)
print(f"Pages: {len(pages)} | Nonempty: {sum(bool(p['text']) for p in pages)}")
print(pages[0]["text"][:1200])

Pages: 6 | Nonempty: 6
VISA
 CONSUMER CREDIT CARD AGREEMENT

In this Agreement, “Agreement” means this Consumer Credit Card Agreement. “Disclosure” means the Credit Card
Account Opening Disclosure. The Account Opening Disclosure is incorporated into this Consumer Credit Card Agreement
and is part of the Agreement. In this Agreement the words "you," “your,” and "yours" mean each and all of those who
agree to be bound by this Agreement; "card" means the Visa credit card and any duplicates, renewals, or substitutions the
Credit Union issues to you; "account" means your Visa credit card line-of-credit account with the Credit Union; and "Credit
Union" means the Credit Union whose name appears on this Agreement or anyone to whom the Credit Union transfers
this Agreement.
1. USING YOUR ACCOUNT — If you are approved for an account, the Credit Union will establish a line of credit for you.
You agree that your credit limit is the maximum amount (purchases, cash advances, finance charges, plus "o

## 3. Page-aware overlapping chunks
Each chunk stays within one page, so the page citation is reliable. Multi-page reasoning can be added in a later extension.

In [7]:
def chunk_page(text, size=3200, overlap=350):
    assert 0 <= overlap < size
    chunks, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            split = text.rfind("\n", start + size // 2, end)
            if split > start:
                end = split + 1
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(start + 1, end - overlap)
    return [c for c in chunks if c]

chunks = []
for page in pages:
    for idx, text in enumerate(chunk_page(page["text"], CHUNK_CHARS, OVERLAP_CHARS)):
        chunks.append({"page": page["page"], "chunk_id": idx, "context": text})
print("Chunks:", len(chunks))
pd.DataFrame([{"page": x["page"], "chunk_id": x["chunk_id"], "characters": len(x["context"])} for x in chunks])

Chunks: 13


,page,chunk_id,characters
0,1,0,3187
1,1,1,3068
2,2,0,3082
3,2,1,3195
4,2,2,889
5,3,0,3199
6,3,1,2966
7,4,0,3107
8,4,1,3155
9,4,2,494


## 4. LLM generation with evidence
The model returns structured JSON. Every answerable QA must contain a verbatim supporting quote from its chunk. For unanswerable QA, the answer is `null` and evidence is empty.

In [8]:
SYSTEM_PROMPT = """You are a meticulous synthetic QA dataset author for document-grounded retrieval evaluation.
Use ONLY the supplied excerpt. Do not use external knowledge. Do not infer missing terms from standard credit card practices.
For each answerable item, copy a SHORT VERBATIM evidence quote from the excerpt that directly supports the answer.
The answer must be fully supported by the quoted evidence. Do not invent APRs, fee amounts, or terms specified only in an absent Disclosure.
Return a JSON object with one key, 'qa_pairs', containing a list of objects. Each object has:
question (string), answer (string or null), evidence (string), category (one of factual, numerical, reasoning, unanswerable).
For unanswerable questions, answer must be null and evidence must be empty; questions should be natural and relevant to the excerpt but cannot be answered from the excerpt.
Avoid repeated or ambiguous questions. Avoid questions requiring information from other pages.
"""

def generate_chunk_qa(chunk, n=6, unanswerable=True):
    desired = "Include exactly one unanswerable question." if unanswerable else "All questions must be answerable."
    prompt = f"""Generate up to {n} high-quality QA pairs for this page excerpt.
Include factual, numerical (when applicable), and reasoning questions. {desired}
If fewer supported questions exist, generate fewer rather than inventing facts.
Page: {chunk['page']}

EXCERPT:
{chunk['context']}"""
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                temperature=0.2,
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": SYSTEM_PROMPT},
                          {"role": "user", "content": prompt}],
            )
            result = json.loads(response.choices[0].message.content)
            if not isinstance(result.get("qa_pairs"), list):
                raise ValueError("Missing qa_pairs array")
            return result["qa_pairs"]
        except Exception as exc:
            if attempt == MAX_RETRIES - 1:
                print(f"Failed page {chunk['page']}, chunk {chunk['chunk_id']}: {exc}")
                return []
            time.sleep(2 ** attempt)
    return []

## 5. Run generation
For a low-cost test, change `chunks_to_process = chunks` to `chunks[:2]`.

In [9]:
chunks_to_process = chunks
raw_records = []
for chunk in tqdm(chunks_to_process, desc="Generating QA"):
    qa_pairs = generate_chunk_qa(chunk, QUESTIONS_PER_CHUNK, GENERATE_UNANSWERABLE)
    for qa in qa_pairs:
        if not isinstance(qa, dict):
            continue
        raw_records.append({
            "question": qa.get("question"),
            "answer": qa.get("answer"),
            "evidence": qa.get("evidence", ""),
            "category": qa.get("category", ""),
            "source_file": PDF_PATH.name,
            "page": chunk["page"],
            "chunk_id": chunk["chunk_id"],
            "context": chunk["context"],
        })
print("Raw QA count:", len(raw_records))

Generating QA: 100%|██████████| 13/13 [00:45<00:00,  3.52s/it]

Raw QA count: 72


## 6. Evidence validation and deduplication
This deterministic check catches missing or fabricated evidence quotes; it does **not** prove that an answer logically follows from the evidence. Review retained records manually or add an independent judge.

In [10]:
def normalize(s):
    return re.sub(r"\s+", " ", str(s or "")).strip().casefold()

def validate_record(r):
    question = r.get("question")
    answer = r.get("answer")
    evidence = r.get("evidence")
    category = r.get("category")
    if not isinstance(question, str) or len(question.strip()) < 10:
        return False, "invalid_question"
    if category not in {"factual", "numerical", "reasoning", "unanswerable"}:
        return False, "invalid_category"
    if category == "unanswerable":
        if answer is not None or normalize(evidence):
            return False, "unanswerable_has_answer_or_evidence"
        return True, "passed_unanswerable_format"
    if not isinstance(answer, str) or not answer.strip():
        return False, "missing_answer"
    if not isinstance(evidence, str) or len(evidence.strip()) < 8:
        return False, "missing_evidence"
    if normalize(evidence) not in normalize(r["context"]):
        return False, "evidence_not_in_context"
    return True, "passed_quote_check"

accepted, rejected, seen = [], [], set()
for r in raw_records:
    ok, reason = validate_record(r)
    r = {**r, "validation": reason}
    if not ok:
        rejected.append(r)
        continue
    key = normalize(r["question"])
    if key in seen:
        rejected.append({**r, "validation": "duplicate_question"})
        continue
    seen.add(key)
    r["id"] = hashlib.sha256((r["source_file"] + "|" + r["question"]).encode()).hexdigest()[:16]
    accepted.append(r)
print("Accepted:", len(accepted), "Rejected:", len(rejected))
print("Categories:", dict(Counter(x["category"] for x in accepted)))
print("Rejection reasons:", dict(Counter(x["validation"] for x in rejected)))

Accepted: 70 Rejected: 2
Categories: {'factual': 40, 'numerical': 7, 'reasoning': 10, 'unanswerable': 13}
Rejection reasons: {'evidence_not_in_context': 2}


## 7. Inspect a sample

In [11]:
columns = ["id", "category", "page", "question", "answer", "evidence", "validation"]
pd.DataFrame(accepted)[columns].head(15) if accepted else pd.DataFrame(columns=columns)

,id,category,page,question,answer,evidence,validation
0,a551b5b50858e6e4,factual,1,What does the term 'Agreement' refer to in thi...,this Consumer Credit Card Agreement,"In this Agreement, “Agreement” means this Cons...",passed_quote_check
1,bcbdaf96a985ae2d,factual,1,What is the consequence of using a convenience...,it will be posted to your account as a cash ad...,"If you use a convenience check, it will be pos...",passed_quote_check
2,ced2645aea56c6d7,numerical,1,How long is a written stop payment order effec...,only for six (6) months,Written stop payment orders are effective only...,passed_quote_check
3,a95d3ab670dd743b,factual,1,Can the Credit Union refuse to pay a convenien...,"Yes, the Credit Union reserves the right to re...",We reserve the right to refuse to pay a conven...,passed_quote_check
4,627ce3930274727b,reasoning,1,What happens if a stop payment request is made...,such a request will expire after 14 days unles...,"If permitted, you may make a stop payment requ...",passed_quote_check
5,24142e87652f5969,unanswerable,1,What is the APR for purchases on this credit c...,None,,passed_unanswerable_format
6,82e58ae72a09983f,factual,1,What must you do if the credit union re-credit...,You agree to sign a statement describing the d...,You agree to sign a statement describing the d...,passed_quote_check
7,e5ff262accd4ca99,factual,1,What is required to obtain cash advances from ...,"To obtain cash advances from an ATM, you must ...","To obtain cash advances from an ATM, you must ...",passed_quote_check
8,42d6dbd0239bae0e,reasoning,1,Are you responsible for charges incurred from ...,Even if you use your card for an illegal trans...,Even if you use your card for an illegal trans...,passed_quote_check
9,c98a345d47a8b30c,factual,1,What types of transactions are considered cash...,Such transactions will be posted to your accou...,Such transactions will be posted to your accou...,passed_quote_check


## 8. Export JSONL and CSV
JSONL retains full context for each item. CSV is convenient for spreadsheet inspection. Rejected records are saved separately for debugging.

In [12]:
def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

jsonl_path = OUTPUT_DIR / "synthetic_qa.jsonl"
csv_path = OUTPUT_DIR / "synthetic_qa.csv"
rejected_path = OUTPUT_DIR / "rejected_qa.jsonl"
summary_path = OUTPUT_DIR / "generation_summary.json"
write_jsonl(jsonl_path, accepted)
write_jsonl(rejected_path, rejected)
pd.DataFrame(accepted, columns=["id", "question", "answer", "category", "evidence", "source_file", "page", "chunk_id", "context", "validation"]).to_csv(csv_path, index=False)
summary = {
    "source": PDF_PATH.name,
    "model": MODEL,
    "pages": len(pages),
    "chunks_processed": len(chunks_to_process),
    "raw_count": len(raw_records),
    "accepted_count": len(accepted),
    "rejected_count": len(rejected),
    "categories": dict(Counter(x["category"] for x in accepted)),
    "limitations": [
        "Quote matching does not establish semantic entailment.",
        "Unanswerability is model-generated and requires manual verification against the full PDF.",
        "The referenced Account Opening Disclosure is not included in this PDF.",
        "Chunks are page-local; this version does not create cross-page multi-hop questions."
    ]
}
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved:")
for p in (jsonl_path, csv_path, rejected_path, summary_path):
    print(" -", p.resolve())

Saved:
 - /home/coder/cibc1-yujia/CIBC1/implementations/qa_text_generation/synthetic_qa_output/synthetic_qa.jsonl
 - /home/coder/cibc1-yujia/CIBC1/implementations/qa_text_generation/synthetic_qa_output/synthetic_qa.csv
 - /home/coder/cibc1-yujia/CIBC1/implementations/qa_text_generation/synthetic_qa_output/rejected_qa.jsonl
 - /home/coder/cibc1-yujia/CIBC1/implementations/qa_text_generation/synthetic_qa_output/generation_summary.json


## 9. Quality-control checklist

Before using this dataset for training or RAG evaluation:

1. Read a random sample of generated questions and verify each answer against the **full PDF**, not just its chunk.
2. Verify that every `unanswerable` question truly cannot be answered elsewhere in the PDF.
3. Review numerical questions for units, conditions, exceptions, and time periods.
4. Keep the original source PDF and evidence/page metadata for auditability.
5. For benchmarking, avoid placing paraphrases of the same question in both train and test sets.
6. Consider adding a second-model entailment check for stronger filtering.

**No API calls are made until you run the generation cell.**